# Inverted Pendulum

Created by Alexander Böhmer

Questions to **alexander.boehmer@oeaw.ac.at** 
or **https://github.com/alboeh**

**Important Note**: You have to change nothing explicitely in the code except for the parts which where labeled with `#TODO` in the exercises!

The whole Skript contains 3 parts, where only the exercises have to be solved!
If you are not interested in the physics and how I implemented the simulation, I suppose you to jump directly to part 2. But don't forget to run all code in the given order.

1) Physics and helper functions (just let it run without further notice)
2) Examples
    - 1: Free pendulum
    - 2: Force-triggered pendulum
3) Exercises
    - 1: PID control
    - 2: LQR control
    - 3: Competition
    - Bonus: LQI control


# 1) Physics and helper functions

**Packages**

I kept it simple, so you can also run the code at your own machine. I only used numpy, scipy and matplotlib.

In [ ]:
# If you do not have, untoggle and install
# pip install numpy matplotlib scipy ipython jupyter

In [ ]:
# Packages etc
import numpy as np                              # General math
import matplotlib.pyplot as plt                 # For plotting
from matplotlib.animation import FuncAnimation  # For animation
from IPython.display import HTML, display       # For inline animation in this notebook
from scipy.linalg import solve_continuous_are   # For solving ricatti equation

# Necessary for simulation to be run as HTML
%matplotlib inline

# just for calculation
rad2deg = 180.0/np.pi
deg2rad = np.pi/180.0

**Pendulum physics**

Here I use a "self-written" numerical method (euler) to solve, so I don't need external solvers etc for simplicity. 
You don't have to fully get the derivation of this step, just keep in mind: We are doing a step-by-step calculation of the states, similarly how the real control will work.

In [ ]:
# Pendulum parameters
g = 9.81    # Gravitation
l = 0.235    # Pendulum vertical arm length
r = 0.11    # Pendulum radial arm length
# Total mass i measures was 20.8 grams
m = 0.0208 * l/(r+l)     # Vertical arm mass (Full arm is l+r, but we only take into account vertical l part)
b = 2e-4    # Rotational damping factor
J = 1 / 3 * m * l**2    # Mass inertia vertical arm

# Maximum acceleration of Stepper without slipping
u_max = 3   # rad/s^2 estimated from manual of the real stepper

# "Make a step", so getting the new state from the old state after one dt timestep considering the actions
def step(state, action, dim, dt):

    # Read state and action
    if dim == 4:
        theta, dtheta, phi, dphi  = state          # current state, all in rad
    if dim == 5:
        theta, dtheta, phi, dphi, z  = state          # current state, all in rad

    u = action                                 # Rotational acceleration of stepper (to be run in this timestep by stepper motor)

    # Definition of the differential equation
    ddphi = u #/ J_r             # Acceleration of the stepper is direcly set by u
    ddtheta = (-m*r*l/2*ddphi + m*l/2*(r+l/2*np.sin(theta))*np.cos(theta)*dphi**2+m*g*l/2*np.sin(theta)-b*dtheta) / J    # Angular acceleration nonlinear ODE
    
    # Semi-implicit Euler update
    dtheta_new = dtheta + dt * ddtheta
    theta_new  = theta  + dt * dtheta_new
    dphi_new   = dphi   + dt * ddphi
    phi_new    = phi    + dt * dphi_new

    if dim == 4:
        return np.array([theta_new, dtheta_new, phi_new, dphi_new])
    
    if dim == 5: # LQI
        z -= phi_new * dt                    # New for LQI: Get integral
        z_new = np.clip(z, -10, 10)
    
        return np.array([theta_new, dtheta_new, phi_new, dphi_new, z_new])


**Simulation**

In [ ]:
# Simulation parameters
dt = 0.01           # Step size (smaller is numerically more stable, but needs more time to calculate)
time_total = 5      # Total time
steps = int(time_total / dt)    # Number of simulation steps

# For plotting we need to save the states over time

time = np.linspace(0, time_total, steps)

# Simulation for whole time span
def simulation(initial_state, actor_function, dim=4):

    states_log = np.zeros((steps, dim))
    action_log = np.zeros(steps)

    # Set initial state
    states_log[0, :] = initial_state[:dim]
    action_log[0] = 0

    # Loop over all time steps
    for i in range(steps-1):
        
        # Next timestep
        t = i * dt

        # Printout for debugging (toggle if needed)
        #print(states_log[i,:], action_log[i])

        # Get next state with previous action
        states_log[i+1,:] = step(states_log[i,:], action_log[i], dt=dt, dim=dim)

        # actor_function is a function that defines the action (acceleration). It can be "None" (a = 0), "PID" (for PID controller) or ....
        # it must have knowledge about the state and the current time
        # Get new action
        u = actor_function(states_log[i, :], t)  # Define actor function externally, can be just 0 or PID or whatever...
        
        # Limit action on realistic capabilities of the stepper
        action_log[i+1] = np.clip(u, -u_max, u_max)
    return states_log, action_log


**Plots and animation**

In [ ]:
# Tableau 10 color scheme (hex)
tabBlue   = "#4E79A7"
tabGreen  = "#59A14F"
tabBrown  = "#9C755F"
tabOrange = "#F28E2B"
tabYellow = "#EDC948"
tabGray   = "#BAB0AC"
tabRed    = "#E15759"
tabPurple = "#B07AA1"
tabTeal   = "#76B7B2"
tabPink   = "#FF9DA7"

# Plot of the state over time
def show_plots(states, action, theta_set = None, title=None):
    plt.close('all')
    fig, axs = plt.subplots(5, 1, figsize=(8, 6), sharex=True)
    
    labels = [r'$\theta\,[\mathrm{°}]$', r'$\dot{\theta}\,[\mathrm{°/s}]$', r'$\phi\,[\mathrm{°}]$', r'$\dot{\phi}\,[\mathrm{°/s}]$', r'$u\,[\mathrm{°/{s^2}}]$']
    colors = [tabBlue, tabBlue, tabGreen, tabGreen, tabRed]
    
    for i in range(4):
        axs[i].plot(time, states[:, i] * rad2deg, colors[i])
        axs[i].set_ylabel(labels[i])
        axs[i].grid()

    axs[4].plot(time, action[:] * rad2deg, colors[4])
    axs[4].set_ylabel(labels[4])
    axs[4].grid()

    if theta_set is not None:
        axs[0].axhline(y = theta_set, color=tabOrange,linestyle='--')
        
    axs[-1].set_xlim(0, time[-1])
    axs[-1].set_xlabel('Time $t$ [s]')
    fig.suptitle("States and actions over time", fontsize=14)    
    plt.tight_layout()
    if title:
        plt.savefig(title + ".pdf")
    plt.show()

# Plot phasespace (very useful!)
def show_phasespace(states):
    plt.close('all')
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))

    xlabels = [r'$\theta\,[\mathrm{rad}]$', r'$\phi\,[\mathrm{rad}]$']
    ylabels = [r'$\dot{\theta}\,[\mathrm{rad/s}]$', r'$\dot{\phi}\,[\mathrm{rad/s}]$']
    colors = [tabBlue, tabGreen]

    for i in range(2):
        axs[i].plot(states[:, 2*i], states[:, 2*i+1], colors[i])
        axs[i].set_ylabel(ylabels[i])
        axs[i].set_xlabel(xlabels[i])
        axs[i].grid()

    fig.suptitle("Phasespace of angle and position", fontsize=14)    
    plt.tight_layout()
    plt.show()

def show_animation(states, interval=20, fps=60, step=1):
    plt.close('all')

    fig, ax = plt.subplots(figsize=(10, 6), dpi=120)

    # Projektion in y-z (Horizontale=y, Vertikale=z)
    ax.set_xlim(-(r + l) - 0.05, (r + l) + 0.05)
    ax.set_ylim(-l - 0.05,  l + 0.05)
    ax.set_xlabel(r'$x\,[\mathrm{m}]$')
    ax.set_ylabel(r'$z\,[\mathrm{m}]$')
    ax.set_aspect('equal')
    ax.grid()

    cart, = ax.plot([], [], 'ks', ms=6)
    rod,  = ax.plot([], [], '-', lw=4, color=tabOrange)
    pole, = ax.plot([], [], '-', lw=4, color=tabRed)

    def init():
        cart.set_data([], [])
        rod.set_data([], [])
        pole.set_data([], [])
        return cart, rod, pole

    def animate(i):
        theta = states[i, 0]
        phi   = states[i, 2]

        xa = r * np.sin(phi)
        za = 0.0

        xp = xa + l * np.sin(theta) * np.cos(phi)
        zp =      l * np.cos(theta)

        cart.set_data([0], [0])
        rod.set_data([0, xa], [0, za])
        pole.set_data([xa, xp], [za, zp])
        return cart, rod, pole

    frames = range(0, states.shape[0], step)

    ani = FuncAnimation(
        fig, animate,
        frames=frames,
        init_func=init,
        interval=interval,
        blit=True
    )
    plt.close(fig)
    display(HTML(ani.to_jshtml(fps=fps)))
    return  

# 2) Examples

For doing the exercises, you don't have to understand the whole simulation, defined previously. The most important things I will show here in two examples which you can simply click through and let it run. Afterwards you will be able to solve the exercises!

## Example 1: Free pendulum
Here I want to show, how the pendulum behaves, if we don't apply any action on it and just let is fall.

**Initial state**

First of all, we have to define an initial state, which is needed for the simulation to run. The parameters are defined in the presentation. It can be written as a state-vector

$\mathbf{x}_0 = \begin{pmatrix} \theta_0 & \dot{\theta}_0 & \phi_0 & \dot{\phi}_0 \end{pmatrix}^\top$.

$\theta$ is the angle that should be controlled (vertical position of the pendulum). $\phi$ is the stepper motor position. $\dot{\theta}$ and $\dot{\phi}$ are the velocities of each.

The initial angle $\theta_0$ was set to a small non-zero value that the pendulum is not perfectly vertical in the beginning. If you want it to be randomized in the beginning, just toggle the random comment.

In [ ]:
# Initial state for simulation
init_state = np.array([
    0.001, #np.random.uniform(-0.001, 0.001),  # theta (toggle to randomize)
    0.0,                            # theta_dot
    0.0,                           # x
    0.0,                           # x_dot
])

**Actor functions**

Actor functions are giving out the actions of the system, here the force on the cart, where the pendulum is mounted on. In this case, it has only one output, since we have one actor. Actor functions taking state vectors $\mathbf{x}$ (such as the one defined before) and the current time $t$ as an input and therefore return the defined force (here in the first example always $F=0$). In reality, the force is applied by the stepper motor.

In [ ]:
# Define actor function for nothing
def af_0(state, t):     # Input state and time
    u = 0       # define origin acceleration
    return u    # Return acceleration

**Simulate**

If you now pass the initial state and the actor function into the pre-defined `simulation(init_state, actor_function)` function, it will calculate the results.

In [ ]:
# Run simulation
states_0, action_0 = simulation(init_state, actor_function=af_0)

**Plotting and animation**

I prepared three functions, where you can give out some useful stuff, like
- `show_plots()`: Gives you the state variables and actor over time.
- `show_phasespace()`: Useful to analyse system (especially when you worked with phasespace before, otherwise they can be ignored.)
- `show_animation()`: Show the pendulum in motion.

In [ ]:
# Show states and action over time
show_plots(states_0, action_0)

**Important**: Even if we have more states, in reality we are only able to observe (measure) $\theta$ (and $\phi$) with the sensor, not the velocities! Keep that in mind!

In [ ]:
# Show phasespace (physicists know how useful they are)
show_phasespace(states_0)

In [ ]:
# Animation (use carefully, this animation is calculation time consuming (approx 1 min)!)
show_animation(states_0)

## Example 2: Actor-triggered pendulum

Now I want to show, how we would apply a force to the pendulum. Lateron, in the example, we calculate the force at each step. For now on, we just apply a sinusoidal force, just to understand the pendulum behaviour a little more. This example is not yet ment for stabilizing.

In [ ]:
# Second test: Create sinusoidal actor function!
# Therefore, we need a "make" function, which returns the actorfunction with given parameters amplitude and frequency
def make_af_cos(A, f):
    def af_cos(state, t):
        u = A * np.cos(2 * np.pi * f * t)
        return u
    return af_cos

# Make af!
af_cos= make_af_cos(A=10, f=2)

# Let the simulation run!
states_cos, action_cos = simulation(init_state, actor_function=af_cos)

# Show!
show_plots(states_cos, action_cos)
show_phasespace(states_cos)

In [ ]:
# Use only if needed, because of high time consumption!
show_animation(states_cos)

Now we have a running simulation of a pendulum! We must now implement a PID tuner that takes over the actions!

# Exercises

## Exercise 1: PID tuner

**Tasks**
- Find a good tuner and its parameters to stabilize the system!
- Try to stabilize different (non-zero) $\theta_{set}$! What do you observe? At what angle the system turns unstable?

**All tasks have skeleton code and hints prepared below!**

First I give the actor function for the PID tuner. Try to comprehend the basic principle from the lecture!


In [ ]:
# DO NOT CHANGE!
def make_af_pid(theta_set, Kp, Ki, Kd):
    integral = 0.0
    last_error = 0.0

    def pid_actor(state, t):
        nonlocal integral, last_error

        # Get state 
        # keep in mind: we just measure theta in reality, since this is the sensor output y(t) --> We have to work only with theta
        theta = state[0]
        error = theta_set - theta

        integral += error * dt
        integral = np.clip(integral, -5.0, 5.0)  # anti-windup: Prevents extreme integral values for long integration times

        derivative = (error - last_error) / dt
        last_error = error

        u = -(Kp * error + Ki * integral + Kd * derivative)     # Actor needs to work against the error!
        return u

    return pid_actor

**TODO**: Find good PID parameters!

You can use "trial-and-error" method **OR** ZN-method from the lecture. (ZN-method not working so good here in my experience)!

Tipp for tuning with "trial-and-error":

1) Set all K to 0
2) Start tuning up only P till stable, but not too much swing up
3) Start tuning up D so that oscillations are minimizing
4) If wanted, fine-tune with I, but this could make it also more instable again
5) Fine-tune all parameters

**Hint**: For better understanding of your parameters, try the combinations: P, PI, PD (or also uncommon ones like only D, I or DI)!

In [ ]:
# Set the desired theta!
theta_set = 0.0   # Desired theta is vertical!

# TODO: Tune PID by changing Kp, Ki and Kd
Kp = ...
Ki = ...
Kd = ...
af_pid = make_af_pid(theta_set=theta_set, Kp=Kp, Ki=Ki, Kd=Kd)

# Let the simulation run!
states_pid, action_pid = simulation(init_state, actor_function=af_pid)
show_plots(states_pid, action_pid, theta_set = theta_set, title="PID controller inverted pendulum")
show_phasespace(states_pid)

In [ ]:
# Animation
show_animation(states_pid)

**Discussions**

What did you observe? Which parameters were most sensitive? Which not? What happened to the other state variables? You observed any offsets? At what initial angle of $\theta_0$ is the limit of stability? 

## Exercise 2: LQR tuner

The PID is the *“simple and quick”* solution for many problems, where offsets and vibrations are acceptable or for systems that are too hard to model. You may also experience, that the position $\phi$ drifts over time although the angle $\theta$ can be stabilized.

One solution would be to implement a second PID to control the position similarly. However, $\phi$ and $\theta$ influence each other directly, and these cross-couplings make multi-state systems hard to tune with PIDs. A better solution are **state-space controllers** (SSCs), as explained in the lecture. One of the most common subtype of SSCs are **LQR tuners**. This tuner is not only capable of controlling many states of the system but also find the optimal solution (regarding the necessary motion and control power), why it is also called an optimal tuner.

The downside of SSCs is, that we need a model of the underlying system, which we gonna derive in the first task. For control theorists this is usually the main part of their work and the hardest part for complicated systems.

**Tasks**

- 2.1) Find the state-space representation 

- 2.2) Check stability 

- 2.3) Check controllability and observability

- 2.4) Tune LQR

**All tasks have skeleton code and hints prepared below!**

### 2.1) State-space representation

Take pen and paper and determine matrices $\mathbf{A}$ and $\mathbf{B}$ such that $\dot{\mathbf{x}} = \mathbf{A}\,\mathbf{x} + \mathbf{B}\,u$.

**Hints**:
- The state vector is
  $\mathbf{x} = \begin{pmatrix} \theta & \dot{\theta} & \phi & \dot{\phi} \end{pmatrix}^\top$
  and $\dot{\mathbf{x}}$ is its time derivative.  
  The input $u$ is the rotational acceleration on the stepper motor and therefore scalar.  
- Because $\mathbf{x}\in\mathbb{R}^4$, the matrices to find are from dimension $\mathbf{A}\in\mathbb{R}^{4\times4}$ and $\mathbf{B}\in\mathbb{R}^{4\times1}$.
- Rewrite the system **line by line** (4 equations) before assembling the matrices. 
  Two of them are simply identities with $\frac{\text{d}}{\text{d}t}\theta=\dot{\theta}$ and $\frac{\text{d}}{\text{d}t}\phi=\dot{\phi}$.
  The third is directly given by the input $\ddot{\phi}=u$.
- The last equation is the underlying differential equation

  $J\,\ddot{\theta} = \frac{mgl}{2}\sin{\theta} - b\,\dot{\theta}-\frac{mrl}{2}\cos{\theta}\,u\,\textcolor{red}{+\frac{ml}{2}(r+\frac{l}{2}\sin{\theta})\cos{\theta}\,\dot{\phi}^2}$

  depending on the constants of gravitation $g$, mass $m$, damping coefficient $b$, the pendulum vertical length $l$ and radial length $r$.
- The mass inertia is given as $J=\frac{1}{3}ml^2$.
- Find the linearized system at $\mathbf{x}_0 = \begin{pmatrix} 0 & 0 & 0 & 0 \end{pmatrix}^\top$ and $u_0=0$ with first-order Taylor approximations (as shown in the lecture).
- The red part in the formula vanishes in first order approximation: $\textcolor{red}{\frac{ml}{2}(r+\frac{l}{2}\sin{\theta})\cos{\theta}\,\dot{\phi}^2\approx0}$. Try out yourself, if you want!
- The first-order Taylor approximations (linearizations) for small angles $\alpha$ are $\sin(\alpha) \approx \alpha$ and $\cos(\alpha) \approx 1$.
- If you correctly rearranged all 4 equations, you will be able to rewrite those in matrix form! 
- **If you really hate math**, you may skip the derivation and obtain the matrices from the tutor (with tears in one's eyes).

In [ ]:
# TODO: Fill out matrices! (Parameters m, g, b, l, r are already pre-defined, just use them directly as variables!)
A_LQR = np.array([
    [..., ..., ..., ...],
    [..., ..., ..., ...],
    [..., ..., ..., ...],
    [..., ..., ..., ...]
])

B_LQR = np.array([
    [...],
    [...],
    [...],
    [...]
])

### 2.2) Check stability

Calculate eigenvalues of $\mathbf{A}$ and interpret the results.
  - What can you say about damping and oscillations?
  - We can add a clamp with additional mass at the end of the pendulum. Will this increase or reduce stability? Calculate!
  - For the "hanging" state of the pendulum ($\theta=\pi$), you would just need to flip the sign of $\sin$ and $\cos$ parts, the rest will be identical as before. How does this change the dynamic?

You can use python to do it numerically, if you don't want to do it "by hand".  

In [ ]:
# TODO: Use numpy to calculate eigenvalues of A
eigen = ...
print(eigen)

In [ ]:
# TODO: Change m to m_heavier in your A matrix (copy the rest). How do the eigenvalues change?

m_heavier = 100 * m     # Heavier mass

# TODO: Copy and paste you previous A matrix here but just change m to m_heavier
A_heavier = np.array([
    [..., ..., ..., ...],
    [..., ..., ..., ...],
    [..., ..., ..., ...],
    [..., ..., ..., ...]
])
eigen_heavier = ...
print(eigen_heavier)

In [ ]:
# TODO: Calc eigenvalues of A matrix in "hanging state"

# TODO: Copy and paste you previous A matrix here but just change respective signs
A_hanging = np.array([
    [..., ..., ..., ...],
    [..., ..., ..., ...],
    [..., ..., ..., ...],
    [..., ..., ..., ...]
])

eigen_hanging = ...
print(eigen_hanging)

### 2.3) Observability and controllability

Get control matrix $\mathcal{C}$ and observer matrix $\mathcal{O}$ and determine the rank. What can we say about controllability and observability?
  - For that, you also need the output matrix $\mathbf{C}$. Since we can measure both angles $\theta$ and $\phi$, we can define this as
  $\mathbf{C} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \end{pmatrix}$, where the output has two states $\mathbf{y} = \begin{pmatrix} \theta & \phi \end{pmatrix}^\top$.
  - You can do it by hand, but python is much faster! 

In [ ]:
# TODO: Interpret the results!

# Controllability
CON = np.hstack([
    B_LQR,
    A_LQR @ B_LQR,
    (A_LQR @ A_LQR) @ B_LQR,
    (A_LQR @ A_LQR @ A_LQR) @ B_LQR
])

rank_CON = np.linalg.matrix_rank(CON)
print("CON=\n", CON)
print("rank(CON)=", rank_CON)

In [ ]:
# TODO: Use the given output matrix C and get the observer matrix. Interpret results.

# Observability

C_LQR = np.array([[1, 0, 0, 0], [0, 0, 1, 0]], dtype=float)

# TODO: Write observer matrix, similar to the CON matrix before. 
OBS = np.vstack([
    C_LQR,
    ...,
    ...,
    ...
])

rank_OBS = np.linalg.matrix_rank(OBS)
print("OBS=\n", OBS)
print("rank(OBS)=", rank_OBS)

### 2.4) LQR tuning

We need to choose good weight matrices $\mathbf{Q}$ and $\mathbf{R}$ for tuning.
  - $\mathbf{Q}$ weights the state errors and $\mathbf{R}$ weights the control error.
  - Start with $1$ in all elements as starting values **OR** use the Bryson method from the lecture.
  - Vary the weights, observe and optimize. (Hint: change initial state by a small offset to see the behavior)
  - Keep in mind: Deviations from which states you want to punish hard? Which are less important? 
  - Also check if the closed Loop $\mathbf{A}-\mathbf{B}\mathbf{K}$ is now stable!

In [ ]:
# TODO: Vary Q and R inputs and see what it changes!

Q_LQR = np.diag([1, 1, 1, 1])   # State cost
R_LQR = np.array([[1]])       # Control cost

In [ ]:
# DO NOT CHANGE

# Solve Riccati equation
P_LQR = solve_continuous_are(A_LQR, B_LQR, Q_LQR, R_LQR)

# LQR-Gain vector
K_LQR = np.linalg.inv(R_LQR) @ B_LQR.T @ P_LQR
print(K_LQR)

In [ ]:
# TODO: Vary initial state (minimally) to see behavior
init_state_LQR = np.array([
    0.01, #np.random.uniform(-0.01, 0.01),  # theta (toggle to randomize)
    0.0,                            # theta_dot
    0.0,                           # phi
    0.0,                           # phi_dot
])

In [ ]:
# DO NOT CHANGE

# Run LQR

# Make actor function for lqr
def make_af_lqr(K):
    def lqr_actor(state, t):
        u = -K @ state
        return u[0]
    return lqr_actor

# Run simulation
af_lqr = make_af_lqr(K_LQR)
states_lqr, action_lqr = simulation(init_state_LQR, af_lqr)

# Results!
show_plots(states_lqr, action_lqr, title="LQR controller inverted pendulum")
show_phasespace(states_lqr)

In [ ]:
# Animation (run only if necessary, needs a lot of time to render)
show_animation(states_lqr)

In [ ]:
# TODO: Check how stability behaves with closed loop!
A_closed = A_LQR-B_LQR*K_LQR
eigen_closed_LQR = ...
print(eigen_closed_LQR)

## Exercise 3: Competition 

Isn't it nice, that we can now in principle stabilize any arbitrary state? (Under the restriction, that "we cannot make a Porsche out of a VW Beetle", of course). 

**Tasks**

- I added a line to the actor function below, that changes the x position (e. g. $\phi_{ref}=30°$) after $1$ second of simulation time. Let the simulation run and check, what it does! Optimize now Q and R and do some testing. What do these parameters affect in the motion? This task is best to train for the competition.
- Finally: find the best $\mathbf{Q}$ and $\mathbf{R}$ values, that you want to use in the competition. Write them down! You have one "practice" try on the real pendulum before the competition! 

**All tasks have skeleton code and hints prepared below!**

In [ ]:
# DO NOT CHANGE

# New "step" actor function
def make_af_lqr_step(K):
    def lqr_actor(state, t):

        # TODO: add more lines here, if you want! (Angle in rad)
        phi_ref = 30 * deg2rad if t > 1.0 else 0.0

        theta, theta_dot, phi, phi_dot= state
        err = np.array([theta, theta_dot, phi - phi_ref, phi_dot])
        u = -K @ err

        return u[0]
    return lqr_actor

In [ ]:
# TODO: Vary to see different behaviors from different initial states
init_state_COM = np.array([
    -0.01, #np.random.uniform(-0.01, 0.01),  # theta (toggle to randomize)
    0.0,                            # theta_dot
    0.0,                           # phi
    0.0                           # phi_dot
])

In [ ]:
# TODO: Find best Q and R matrices for the competition! Fine tune best you can :) 
# As start values you can use the ones from the previous exercise.

# Punishing [theta, theta_dot, pos, vel, int]
Q_COM = np.diag([1, 1, 1, 1])   # ...
R_COM = np.array([[1]])       # ...

# Solve Riccati equation
P_COM = solve_continuous_are(A_LQR, B_LQR, Q_COM, R_COM)

# LQR-Gain vector
K_COM = np.linalg.inv(R_COM) @ B_LQR.T @ P_COM
print(K_COM)

In [ ]:
# Create actor and run simulation
af_step = make_af_lqr_step(K_COM)
states_step, action_step = simulation(init_state_COM, af_step)

# Show results
show_plots(states_step, action_step, title="LQR controller with step")
show_phasespace(states_step)

In [ ]:
# Animation
show_animation(states_step)

## BONUS Exercise: LQI controller

"Real" pendulums have disturbances which cannot be foreseen in simulation. If they are strong, both PID and LQR have problems keeping certain states at the desired point. For the real pendulums position $\phi$ it is particularly critical in a sense, that all positions (full 360° rotation) are strongly dependend to the tilt of the underground. Even small disturbances would move the stabilized pendulum then slowly to undersired positions $\phi$. Therefore we want to "play safe" by implementing an integrator, that helps us to keep $\phi$ to $0°$. This is sometimes calles **LQI** control.

**Tasks**

- 3.1) Include integrator state.

- 3.2) Tune LQI 

**All tasks have skeleton code and hints prepared below!**

### B.1) Integrator state

Based on the previous defined SSR, extend $\mathbf{A}$ and $\mathbf{B}$ by an integrator state $z$.
 - The integrator is defined as $z=\int r_0-\phi\, \text{d}t$ with $r_0=0$. (You need to get $\dot{z}$!)
 - Yes, the solution is super simple and only contains $0$ and $1$!
 - Use the lecture to understand.
 - Except the new integrator state, the rest of the matrices stay identical.

In [ ]:
# TODO: Give new extended matrices A and B! Copy the previous A and B.
A_LQI = np.array([
    [..., ..., ..., ..., 0],
    [..., ..., ..., ..., 0],
    [..., ..., ..., ..., 0],
    [..., ..., ..., ..., 0],
    [..., ..., ..., ..., ...]
])

B_LQI = np.array([
    [...],
    [...],
    [...],
    [...],
    [...]
])

### B.2) Tune LQI

In [ ]:
# TODO: Add the additional state to Q and punish as desired!

# Punishing [theta, theta_dot, pos, vel, int]
Q_LQI = np.diag([1, 1, 1, 1, 1])   # ...
R_LQI = np.array([[1]])       # ...

# Solve Riccati equation
P_LQI = solve_continuous_are(A_LQI, B_LQI, Q_LQI, R_LQI)

# LQR-Gain vector
K_LQI = np.linalg.inv(R_LQI) @ B_LQI.T @ P_LQI
print(K_LQI)

In [ ]:
# TODO: Vary as desired
init_state_LQI = np.array([
    0.01, #np.random.uniform(-0.01, 0.01),  # theta (toggle to randomize)
    0.0,                            # theta_dot
    0.0,                           # phi
    0.0,                           # phi_dot
    0.0                        # integrator state
])

In [ ]:
# DO NOT CHANGE

# Run LQI

# Actor function for LQI indentical to LRR
af_LQI = make_af_lqr(K_LQI)
states_LQI, action_LQI = simulation(init_state_LQI, af_LQI, dim=5)

# Results!
show_plots(states_LQI, action_LQI, title="LQI controller inverted pendulum")
show_phasespace(states_LQI)

In [ ]:
show_animation(states_LQI)

# More ideas

How fast is the state changed? What's the reason for that? Have fun, trying different things with this simulation. Here's so much more to explore! Examples:
- What happens, when the x_ref is too far away?
- Try extreme Q and R settings!
- Try to set $\theta$ to a non-zero value!
- **Train a Reinforcement Learning (ML) framework with this simulation and feed it to the real pendulum!**

Hope you had fun :) Thanks!

Questions to **alexander.boehmer@oeaw.ac.at**